# Classificador de Textos: Humano vs IA
### DistilBERT fine-tuned — OpenAI, Meta, Google, Anthropic, Humano

## Celula 1 - Instalacao

In [ ]:
!pip uninstall datasets evaluate -y
!pip install transformers==4.40.0 tokenizers==0.19.0 scikit-learn pandas numpy torch matplotlib seaborn accelerate

## Celula 2 - Diagnostico

In [ ]:
import sys
print('Python:', sys.executable)
import torch
print('PyTorch:', torch.__version__)
import transformers
print('Transformers:', transformers.__version__)
from transformers.utils import is_torch_available
print('PyTorch visivel para Transformers:', is_torch_available())
print('GPU disponivel:', torch.cuda.is_available())

## Celula 3 - Imports

In [ ]:
import os
import re
import time
import pickle
import unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('Imports OK!')

## Celula 4 - Configuracoes

In [ ]:
SEED          = 42
MAX_LENGTH    = 128
BATCH_SIZE    = 32        # roberta e maior, precisa de menos batch
EPOCHS        = 8         # converge mais rapido que distilbert
LEARNING_RATE = 2e-5      # roberta responde bem a LR um pouco mais alto
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
DROPOUT       = 0.1
MODEL_NAME    = 'roberta-base'
PATIENCE      = 3
NUM_LABELS    = 5

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## Celula 5 - Funcoes de limpeza

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s.,!?;:\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_light(text):
    text = str(text)
    text = unicodedata.normalize('NFKC', text)
    replacements = {
        '\u2018': "'", '\u2019': "'",
        '\u201c': '"', '\u201d': '"',
        '\u2013': '-', '\u2014': '-',
        '\u00a0': ' ',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def strip_json_edge_artifacts(text):
    text = str(text).strip()
    edge_chars = set('"\' ()[]{}')
    while text and text[0] in edge_chars:
        text = text[1:].lstrip()
    while text and text[-1] in edge_chars:
        text = text[:-1].rstrip()
    return text

print('Funcoes de limpeza OK!')

## Celula 6 - Carregar e preparar dados de treino

In [ ]:
BASE_DIR   = r'C:\Users\FCCas\OneDrive\Desktop\Uni\4_ano\2_semestre\Aprendizagem Profunda\Trabalho'
SPLIT_DIR  = os.path.join(BASE_DIR, 'split')

# Caminhos dos ficheiros
TRAIN_PATH = os.path.join(SPLIT_DIR, 'train.csv')
VAL_PATH   = os.path.join(SPLIT_DIR, 'val.csv')
TEST_PATH  = os.path.join(SPLIT_DIR, 'test.csv')

# Carregar CSVs
train_df = pd.read_csv(TRAIN_PATH, sep=',')
val_df   = pd.read_csv(VAL_PATH,   sep=',')
test_df  = pd.read_csv(TEST_PATH,  sep=',')

print(f'Treino: {len(train_df)} | Validacao: {len(val_df)} | Teste: {len(test_df)}')

# Limpeza
for df in [train_df, val_df, test_df]:
    df['content'] = df['content'].apply(clean_text)

# Remover linhas com texto muito curto
train_df = train_df[train_df['content'].str.len() > 10].reset_index(drop=True)
val_df   = val_df[val_df['content'].str.len() > 10].reset_index(drop=True)
test_df  = test_df[test_df['content'].str.len() > 10].reset_index(drop=True)

print(f'\nApos limpeza:')
print(f'Treino: {len(train_df)} | Validacao: {len(val_df)} | Teste: {len(test_df)}')
print(f'\nDistribuicao classes (treino):')
print(train_df['model'].value_counts())

# Encoding de labels
le = LabelEncoder()
le.fit(train_df['model'])

train_df['label'] = le.transform(train_df['model'])
val_df['label']   = le.transform(val_df['model'])
test_df['label']  = le.transform(test_df['model'])

print('\nMapeamento:')
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {idx} -> {cls}')

## Celula 7 - Dataset PyTorch e Tokenizer

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_length,
        )
        self.input_ids      = torch.tensor(encodings['input_ids'],      dtype=torch.long)
        self.attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long)
        self.labels         = torch.tensor(list(labels),                dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx],
        }

print('A carregar tokenizer...')
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

print('A criar datasets...')
train_dataset = TextDataset(train_df['content'], train_df['label'], tokenizer)
val_dataset   = TextDataset(val_df['content'],   val_df['label'],   tokenizer)
test_dataset  = TextDataset(test_df['content'],  test_df['label'],  tokenizer)
print('Datasets criados!')

## Celula 8 - Construir modelo

In [ ]:
id2label = {i: c for i, c in enumerate(le.classes_)}
label2id = {c: i for i, c in enumerate(le.classes_)}

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    id2label=id2label,
    label2id=label2id,
).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros totais:     {total:,}')
print(f'Parametros treinaveis: {trainable:,}')

## Celula 9 - Treino

In [ ]:
# ---- Class weights ----
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label'].values),
    y=train_df['label'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print('Class weights:')
for cls, w in zip(le.classes_, class_weights.cpu().numpy()):
    print(f'  {cls}: {w:.4f}')
# -----------------------

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

history        = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1    = -1
patience_count = 0
best_state     = None

def fmt(secs):
    m, s = int(secs // 60), int(secs % 60)
    return f'{m}m{s:02d}s'

print('=' * 60)
print('A iniciar treino...')
print('=' * 60)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    model.train()
    train_loss       = 0.0
    train_preds_all  = []
    train_labels_all = []

    train_bar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Treino]',
                     leave=True, dynamic_ncols=True)

    for batch in train_bar:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        train_preds_all.extend(preds.cpu().numpy())
        train_labels_all.extend(labels.cpu().numpy())
        train_bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = accuracy_score(train_labels_all, train_preds_all)

    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []

    val_bar = tqdm(val_loader, desc=f'Epoch {epoch:02d}/{EPOCHS} [Val]   ',
                   leave=True, dynamic_ncols=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs   = model(input_ids=input_ids, attention_mask=attention_mask)
            loss      = loss_fn(outputs.logits, labels)
            val_loss += loss.item()
            preds     = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            val_bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = accuracy_score(all_labels, all_preds)
    val_f1       = f1_score(all_labels, all_preds, average='macro')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed   = time.time() - start_time
    avg_epoch = elapsed / epoch
    remaining = avg_epoch * (EPOCHS - epoch)

    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | '
          f'Epoch: {fmt(time.time() - epoch_start)} | Restante: ~{fmt(remaining)}')

    if val_f1 > best_val_f1 + 0.001:
        best_val_f1    = val_f1
        patience_count = 0
        best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        print(f'  -> Melhor modelo guardado (F1={best_val_f1:.4f})')
    else:
        patience_count += 1
        print(f'  -> Sem melhoria ({patience_count}/{PATIENCE})')
        if patience_count >= PATIENCE:
            print(f'Early stopping na epoch {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nTreino concluido! Tempo total: {fmt(time.time() - start_time)}')

## Celula 10 - Guardar modelo e tokenizer

In [ ]:
os.makedirs('./best_model', exist_ok=True)
model.save_pretrained('./best_model')
tokenizer.save_pretrained('./best_model')

# Guarda o LabelEncoder para usar na inferencia
with open('./best_model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Modelo guardado em: ./best_model')
print('Ficheiros:')
for fname in os.listdir('./best_model'):
    print(f'  {fname}')

## Celula 11 - Avaliacao no conjunto de teste

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
        preds          = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print('Relatorio de Classificacao (conjunto de teste):')
print(classification_report(all_labels, all_preds, target_names=le.classes_, digits=4))

## Celula 12 - Matriz de confusao

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusao - Conjunto de Teste')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Celula 13 - Curvas de treino

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Loss treino vs validacao
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Treino',    linewidth=2, markersize=6)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Validacao', linewidth=2, markersize=6)
axes[0].set_title('Loss: Treino vs Validacao')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Accuracy treino vs validacao
axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Treino',    linewidth=2, markersize=6)
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Validacao', linewidth=2, markersize=6)
axes[1].set_title('Accuracy: Treino vs Validacao')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Treino - DistilBERT', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Celula 14 - Carregar modelo guardado (sessao nova)
### Corre esta celula se quiseres usar o modelo sem treinar de novo

In [ ]:
# Descomenta estas linhas se estiveres numa sessao nova

# model     = RobertaForSequenceClassification.from_pretrained('./best_model').to(device)
# tokenizer = RobertaTokenizerFast.from_pretrained('./best_model')
# with open('./best_model/label_encoder.pkl', 'rb') as f:
#     le = pickle.load(f)

# print('Modelo carregado!')
# print('Classes:', list(le.classes_))

print('Para carregar o modelo numa sessao nova, descomenta as linhas acima.')

## Celula 15 - Previsoes nos dados de exemplo
### Ficheiro CSV com colunas: ID, Text, Label (Label e opcional)

In [ ]:
EXEMPLOS_PATH = os.path.join(BASE_DIR, 'dataset-exemplos.csv')  # <- fora da pasta split
EXEMPLOS_SEP  = ';'

# 1. Carrega os dados de exemplo
dados_exemplo = pd.read_csv(EXEMPLOS_PATH, sep=EXEMPLOS_SEP)
print(f'Dados de exemplo: {len(dados_exemplo)} linhas')
print(dados_exemplo.head())

# 2. Limpeza
dados_exemplo['Text'] = dados_exemplo['Text'].apply(clean_text_light)
dados_exemplo['Text'] = dados_exemplo['Text'].apply(strip_json_edge_artifacts)
dados_exemplo['Text'] = dados_exemplo['Text'].apply(clean_text)

# 3. Tokenizar
encodings = tokenizer(
    list(dados_exemplo['Text']),
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
)
input_ids      = torch.tensor(encodings['input_ids'],      dtype=torch.long).to(device)
attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long).to(device)

# 4. Previsoes em batches
model.eval()
previsoes      = []
probabilidades = []

with torch.no_grad():
    for i in range(0, len(input_ids), BATCH_SIZE):
        batch_ids   = input_ids[i:i+BATCH_SIZE]
        batch_mask  = attention_mask[i:i+BATCH_SIZE]
        outputs     = model(input_ids=batch_ids, attention_mask=batch_mask)
        probs       = torch.softmax(outputs.logits, dim=-1)
        preds       = torch.argmax(probs, dim=-1)
        for pred, prob in zip(preds.cpu().numpy(), probs.cpu().numpy()):
            previsoes.append(le.classes_[pred])
            probabilidades.append({le.classes_[j]: round(float(prob[j]), 4) for j in range(len(le.classes_))})

# 5. Adiciona previsoes
dados_exemplo['Previsao'] = previsoes

# 6. Accuracy (se existir coluna Label)
if 'Label' in dados_exemplo.columns:
    correct = (dados_exemplo['Label'] == dados_exemplo['Previsao']).sum()
    total   = len(dados_exemplo)
    acc     = correct / total
    print(f'\nAccuracy: {acc:.4f} ({correct}/{total} corretos)')
    print('\nRelatorio de Classificacao:')
    print(classification_report(dados_exemplo['Label'], dados_exemplo['Previsao'], digits=4))

# 7. Mostrar resultados
cols = ['ID', 'Label', 'Previsao'] if 'Label' in dados_exemplo.columns else ['ID', 'Previsao']
print('\nResultados:')
print(dados_exemplo[cols].to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

EXEMPLOS_PATH = os.path.join(BASE_DIR, 'dataset-exemplos.csv')
EXEMPLOS_SEP  = ';'

# 1. Carrega os dados de exemplo
dados_exemplo = pd.read_csv(EXEMPLOS_PATH, sep=EXEMPLOS_SEP)
print(f'Dados de exemplo: {len(dados_exemplo)} linhas')
print(dados_exemplo.head())

# 2. Limpeza
dados_exemplo['Text'] = dados_exemplo['Text'].apply(clean_text_light)
dados_exemplo['Text'] = dados_exemplo['Text'].apply(strip_json_edge_artifacts)
dados_exemplo['Text'] = dados_exemplo['Text'].apply(clean_text)

# 3. Tokenizar
encodings = tokenizer(
    list(dados_exemplo['Text']),
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
)
input_ids      = torch.tensor(encodings['input_ids'],      dtype=torch.long).to(device)
attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long).to(device)

# 4. Previsoes em batches
model.eval()
previsoes      = []
probabilidades = []

with torch.no_grad():
    for i in range(0, len(input_ids), BATCH_SIZE):
        batch_ids  = input_ids[i:i+BATCH_SIZE]
        batch_mask = attention_mask[i:i+BATCH_SIZE]
        outputs    = model(input_ids=batch_ids, attention_mask=batch_mask)
        probs      = torch.softmax(outputs.logits, dim=-1)
        preds      = torch.argmax(probs, dim=-1)
        for pred, prob in zip(preds.cpu().numpy(), probs.cpu().numpy()):
            previsoes.append(le.classes_[pred])
            probabilidades.append({le.classes_[j]: round(float(prob[j]), 4) for j in range(len(le.classes_))})

# 5. Adiciona previsoes
dados_exemplo['Previsao'] = previsoes

# 6. Accuracy e metricas detalhadas
if 'Label' in dados_exemplo.columns:
    correct = (dados_exemplo['Label'] == dados_exemplo['Previsao']).sum()
    total   = len(dados_exemplo)
    acc     = correct / total
    print(f'\nAccuracy: {acc:.4f} ({correct}/{total} corretos)')

    # Classification report
    print('\nRelatorio de Classificacao:')
    print(classification_report(dados_exemplo['Label'], dados_exemplo['Previsao'], digits=4))

    # Accuracy por classe
    print('Accuracy por classe:')
    for classe in sorted(dados_exemplo['Label'].unique()):
        mask = dados_exemplo['Label'] == classe
        c    = (dados_exemplo.loc[mask, 'Label'] == dados_exemplo.loc[mask, 'Previsao']).sum()
        t    = mask.sum()
        print(f'  {classe:<12}: {c/t:.4f} ({c}/{t})')

    # Matriz de confusao
    classes_presentes = [l for l in le.classes_ if l in dados_exemplo['Label'].unique()]
    cm = confusion_matrix(dados_exemplo['Label'], dados_exemplo['Previsao'], labels=classes_presentes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes_presentes, yticklabels=classes_presentes)
    plt.title('Matriz de Confusao - Dados de Exemplo')
    plt.ylabel('Real')
    plt.xlabel('Previsto')
    plt.tight_layout()
    plt.show()

# 7. Mostrar resultados
cols = ['ID', 'Label', 'Previsao'] if 'Label' in dados_exemplo.columns else ['ID', 'Previsao']
print('\nResultados:')
print(dados_exemplo[cols].to_string(index=False))

## Celula 16 - Exportar resultados para CSV

In [ ]:
# Adiciona probabilidades por classe
probs_df = pd.DataFrame(probabilidades)
probs_df.columns = [f'prob_{c}' for c in probs_df.columns]
resultado_final = pd.concat([dados_exemplo.reset_index(drop=True), probs_df], axis=1)

resultado_final.to_csv('previsoes_exemplos.csv', index=False, sep=';')
print('Resultados guardados em: previsoes_exemplos.csv')
print(resultado_final.head())

In [ ]:
# ============================================================
# PREVISOES NOS DADOS DE SUBMISSAO
# Ficheiro: subm2.csv (colunas: ID, Text)
# Output:   subm2_com_labels.csv (colunas: ID, Text, Label)
# ============================================================

SUBMISSAO_PATH = 'subm2.csv'   # <- caminho para o ficheiro do professor
SUBMISSAO_SEP  = ';'           # <- separador do CSV

# 1. Carrega os dados de submissao
df_subm = pd.read_csv(SUBMISSAO_PATH, sep=SUBMISSAO_SEP)
print(f'Dados de submissao carregados: {len(df_subm)} linhas')
print(f'Colunas: {df_subm.columns.tolist()}')
print(df_subm.head())

# 2. Limpeza do texto
df_subm['Text'] = df_subm['Text'].apply(clean_text_light)
df_subm['Text'] = df_subm['Text'].apply(strip_json_edge_artifacts)
textos_limpos   = df_subm['Text'].apply(clean_text).tolist()

# 3. Tokenizar
encodings      = tokenizer(
    textos_limpos,
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
)
input_ids      = torch.tensor(encodings['input_ids'],      dtype=torch.long).to(device)
attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long).to(device)

# 4. Previsoes em batches
model.eval()
previsoes = []

with torch.no_grad():
    for i in range(0, len(input_ids), BATCH_SIZE):
        batch_ids   = input_ids[i:i+BATCH_SIZE]
        batch_mask  = attention_mask[i:i+BATCH_SIZE]
        outputs     = model(input_ids=batch_ids, attention_mask=batch_mask)
        preds       = torch.argmax(outputs.logits, dim=-1)
        for pred in preds.cpu().numpy():
            previsoes.append(le.classes_[pred])

# 5. Adiciona coluna Label
df_subm['Label'] = previsoes

# 6. Mostra distribuicao das previsoes
print('\nDistribuicao das previsoes:')
print(df_subm['Label'].value_counts())

# 7. Guarda o ficheiro final
df_subm.to_csv('subm2_com_labels_MODELO3.csv', index=False, sep=';')
print(f'\nFicheiro guardado: subm2_com_labels.csv')
print(df_subm[['ID', 'Label']].to_string(index=False))